# Alimentation de la fiche produit NBO

Un bloc de code par section du classeur. Chaque bloc affiche les valeurs a recopier
dans les cellules jaunes de la feuille produit.

Les noms de colonnes sont detectes automatiquement (cellule 0b). En cas d'echec,
forcer la colonne dans le dictionnaire `FORCER`.

**Non automatise** : les KPI d'equipement du profiling (multi-equipe, souscription
12 mois) - a saisir manuellement.


## 0a - Configuration


In [ ]:
import pandas as pd, numpy as np, unicodedata
from scipy.stats import spearmanr

# ---- Un dictionnaire par millesime ----
MILLESIMES = {
    "Sep 2025":  {"scores": "scores_sep2025.csv",
                  "detenteurs": "Detenteurs_09_2025.xlsx",
                  "base": "base_sep2025.xlsx"},
    "Mars 2026": {"scores": "scores_mars2026.csv",
                  "detenteurs": "Detenteurs_03_2026.xlsx",
                  "base": "base_mars2026.xlsx"},
    "Juin 2026": {"scores": "scores_juin2026.csv",
                  "detenteurs": "Detenteurs_06_2026.xlsx",
                  "base": "base_juin2026.xlsx"},
}

REF, CUR = "Mars 2026", "Juin 2026"   # millesimes du PSI (section 4)

# ---- Campagne ----
F_CAMPAGNE = "campagne.csv"           # ID + colonne clic (1/0)
MIL_AVANT  = "Mars 2026"              # millesime de score AVANT la campagne
MIL_APRES  = "Juin 2026"              # millesime APRES la fenetre d'observation
# Nouveaux detenteurs = presents dans MIL_APRES et absents de MIL_AVANT.

# ---- Profiling : colonnes du fichier base eligible ----
COLS_PROFIL = {"Age moyen (ans)": "age",
               "Anciennete de la relation (ans)": "anciennete",
               "Part de clients actifs": "actif_3m",
               "Part de clients Mass Affluent": "mass_affluent"}
FILTRE_PARC = None                    # ex. "age < 25" pour Pack Jeunes, sinon None

# ---- Seuil : deduit du flag recommandation (plus petit score recommande) ----
SEUIL_MANUEL = {}                     # ex. {"Sep 2025": 0.157} pour forcer une valeur


## 0b - Detection des colonnes

Ajouter un libelle dans `ALIAS` s'il n'est pas reconnu, ou forcer directement
la colonne dans `FORCER` (cle = nom de fichier).


In [ ]:
ALIAS = {
    "id":    ["identifiant tiers", "id tiers", "idtiers", "identifiant", "num tiers",
              "numero tiers", "id client", "client id", "identifiant client", "tiers"],
    "score": ["score", "score appetence", "appetence", "proba", "probabilite"],
    "reco":  ["recommandation", "recommendation", "reco", "recommande", "flag reco", "top reco"],
    "clic":  ["clic", "click", "a clique", "top clic", "flag clic"],
}

FORCER = {}   # ex. {"Detenteurs_03_2026.xlsx": {"id": "NUM_TIERS_CLIENT"}}

def _norm(s):
    s = unicodedata.normalize("NFKD", str(s)).encode("ascii", "ignore").decode()
    return " ".join("".join(c if c.isalnum() else " " for c in s.lower()).split())

def col(df, role, fichier, obligatoire=True):
    """Retourne le nom reel de la colonne jouant le role demande."""
    if fichier in FORCER and role in FORCER[fichier]:
        return FORCER[fichier][role]
    cols = {c: _norm(c) for c in df.columns}
    for a in ALIAS[role]:                       # correspondance exacte
        for c, n in cols.items():
            if n == a:
                return c
    for a in ALIAS[role]:                       # correspondance partielle
        for c, n in cols.items():
            if a in n:
                return c
    if role == "id" and len(df.columns) == 1:   # fichier a une seule colonne
        return df.columns[0]
    if obligatoire:
        raise KeyError(f"[{fichier}] colonne '{role}' introuvable. "
                       f"Colonnes disponibles : {list(df.columns)}. "
                       f"Ajouter un alias ou renseigner FORCER.")
    return None

def lire(f):
    return pd.read_excel(f) if str(f).endswith((".xlsx", ".xls")) else pd.read_csv(f)


In [ ]:
scores, detenteurs, bases, seuils = {}, {}, {}, {}
detecte = []

for m, cfg in MILLESIMES.items():
    # --- scores ---
    d = lire(cfg["scores"])
    c_id = col(d, "id", cfg["scores"])
    c_sc = col(d, "score", cfg["scores"])
    c_re = col(d, "reco", cfg["scores"])
    d = d.rename(columns={c_id: "ID", c_sc: "SCORE", c_re: "RECO"})[["ID", "SCORE", "RECO"]]
    d["SCORE"] = pd.to_numeric(d["SCORE"], errors="coerce")
    scores[m] = d
    seuils[m] = SEUIL_MANUEL.get(m, d.loc[d.RECO == 1, "SCORE"].min())
    detecte.append((cfg["scores"], f"id={c_id} | score={c_sc} | reco={c_re}"))

    # --- detenteurs ---
    dd = lire(cfg["detenteurs"])
    c = col(dd, "id", cfg["detenteurs"])
    detenteurs[m] = set(dd[c].dropna().unique())
    detecte.append((cfg["detenteurs"], f"id={c}"))

    # --- base eligible ---
    b = lire(cfg["base"])
    c = col(b, "id", cfg["base"])
    bases[m] = b.rename(columns={c: "ID"})
    detecte.append((cfg["base"], f"id={c}"))

print("--- Colonnes detectees ---")
for f, info in detecte:
    print(f"{f:28} {info}")

print("\n--- Volumetrie et controle des cles ---")
for m in MILLESIMES:
    r_sc = scores[m].ID.isin(bases[m].ID).mean()
    r_dt = pd.Series(list(detenteurs[m])).isin(bases[m].ID).mean()
    print(f"{m:10} | scores {len(scores[m]):>7,} | detenteurs {len(detenteurs[m]):>7,} | "
          f"base {len(bases[m]):>7,} | scores en base {r_sc:.1%} | detenteurs en base {r_dt:.1%}")

print("\nSeuils deduits :", {m: round(float(s), 4) for m, s in seuils.items()})


## Sections 2 et 3 - Volumetrie, seuils et couverture


In [ ]:
lignes = []
for m in MILLESIMES:
    d, s = scores[m], seuils[m]
    lignes.append({"Millesime": m, "Seuil": round(float(s), 4),
                   "Clients scores": len(d),
                   "Score = 0": int((d.SCORE == 0).sum()),
                   "0 < Score < Seuil": int(((d.SCORE > 0) & (d.SCORE < s)).sum()),
                   "Score >= Seuil": int((d.SCORE >= s).sum()),
                   "Vrais detenteurs": len(detenteurs[m]),
                   "Base eligible": len(bases[m])})
vol = pd.DataFrame(lignes)
vol["Controle"] = np.where(
    vol[["Score = 0", "0 < Score < Seuil", "Score >= Seuil"]].sum(axis=1) == vol["Clients scores"],
    "OK", "ECART")

print("--- Section 2 (B10:G12) ---")
print(vol.drop(columns=["Base eligible"]).to_string(index=False))

print("\n--- Section 3 (B16:D20) ---")
sec3 = pd.DataFrame({
    "Base eligible": vol["Base eligible"].values,
    "Taux de couverture": (vol["Clients scores"] / vol["Base eligible"]).values,
    "Part de scores nuls": (vol["Score = 0"] / vol["Clients scores"]).values,
    "Taux de selection": (vol["Score >= Seuil"] / vol["Clients scores"]).values,
    "Taux de detention": (vol["Vrais detenteurs"] / vol["Base eligible"]).values,
}, index=list(vol["Millesime"])).T
print(sec3.to_string(float_format=lambda x: f"{x:,.4f}"))


## Section 4 - PSI (tranches figees sur le millesime de reference)


In [ ]:
s_ref = scores[REF].loc[scores[REF].SCORE > 0, "SCORE"]      # zeros exclus
s_cur = scores[CUR].loc[scores[CUR].SCORE > 0, "SCORE"]

bornes = np.unique(np.quantile(s_ref, np.linspace(0, 1, 11)))
bornes[0], bornes[-1] = 0, np.inf

e_ref = pd.cut(s_ref, bornes).value_counts().sort_index()
e_cur = pd.cut(s_cur, bornes).value_counts().sort_index().reindex(e_ref.index, fill_value=0)

psi = pd.DataFrame({"eff_ref": e_ref.values, "eff_cur": e_cur.values,
                    "bornes": e_ref.index.astype(str)})
psi["pct_ref"] = psi.eff_ref / psi.eff_ref.sum()
psi["pct_cur"] = psi.eff_cur / psi.eff_cur.sum()
psi["contrib"] = (psi.pct_cur - psi.pct_ref) * np.log(
    psi.pct_cur.clip(lower=1e-6) / psi.pct_ref.clip(lower=1e-6))
psi = psi.iloc[::-1].reset_index(drop=True); psi.index += 1   # tranche 1 = meilleurs scores

if len(psi) < 10:
    print(f"ATTENTION : {len(psi)} tranches au lieu de 10 - masse de scores identiques "
          f"({s_ref.value_counts().iloc[0]:,} clients sur la valeur la plus frequente).\n")

print(psi.round(4).to_string())
print(f"\nPSI total : {psi.contrib.sum():.4f}")
print(f"\n--- A coller en B32:C41 ---\n"
      f"{psi[['eff_ref','eff_cur']].to_string(index=False, header=False)}")


## Section 5 - Stabilite de la liste a cibler


In [ ]:
mils = list(MILLESIMES)
prec = mils[mils.index(CUR) - 1]

liste_cur  = set(scores[CUR].loc[scores[CUR].SCORE >= seuils[CUR], "ID"])
liste_prec = set(scores[prec].loc[scores[prec].SCORE >= seuils[prec], "ID"])

j = (scores[prec][["ID", "SCORE"]].rename(columns={"SCORE": "s_prec"})
     .merge(scores[CUR][["ID", "SCORE"]].rename(columns={"SCORE": "s_cur"}), on="ID"))
j = j[(j.s_prec > 0) & (j.s_cur > 0)]
rho = spearmanr(j.s_prec, j.s_cur).correlation if len(j) > 30 else float("nan")

print(f"Nb clients >= seuil ({CUR})      : {len(liste_cur):,}")
print(f"dont deja >= seuil ({prec})      : {len(liste_cur & liste_prec):,}")
print(f"Taux de recouvrement             : {len(liste_cur & liste_prec)/max(len(liste_cur),1):.1%}")
print(f"Spearman ({prec} vs {CUR})       : {rho:.2f}  sur {len(j):,} clients communs")


## Section 6 - Controle detenteurs


In [ ]:
d = pd.DataFrame({"ID": list(detenteurs[CUR])}).merge(
        scores[CUR][["ID", "SCORE"]], on="ID", how="left")
s = seuils[CUR]

n_tot   = len(d)
n_fuite = int((d.SCORE >= s).sum())
n_zero  = int((d.SCORE == 0).sum())
n_entre = int(((d.SCORE > 0) & (d.SCORE < s)).sum())
n_abs   = int(d.SCORE.isna().sum())

print(f"Nb de vrais detenteurs ({CUR})    : {n_tot:,}")
print(f"dont score >= seuil (fuite)       : {n_fuite:,}  ({n_fuite/n_tot:.1%})")
print(f"dont score = 0                    : {n_zero:,}  ({n_zero/n_tot:.1%})")
print(f"dont 0 < score < seuil            : {n_entre:,}")
print(f"dont absents du fichier de scores : {n_abs:,}  ({n_abs/n_tot:.1%})   <-- a surveiller")


## Section 7 - Backtest campagne

Nouveaux detenteurs = presents dans `MIL_APRES` et absents de `MIL_AVANT`.


In [ ]:
camp = lire(F_CAMPAGNE)
c_id = col(camp, "id", F_CAMPAGNE)
c_cl = col(camp, "clic", F_CAMPAGNE, obligatoire=False)
camp = camp.rename(columns={c_id: "ID", **({c_cl: "CLIC"} if c_cl else {})})
print(f"Campagne : id={c_id} | clic={c_cl}")

nouveaux = detenteurs[MIL_APRES] - detenteurs[MIL_AVANT]
s = seuils[MIL_AVANT]

c = camp.merge(scores[MIL_AVANT][["ID", "SCORE"]], on="ID", how="left")
c["souscripteur"] = c.ID.isin(nouveaux).astype(int)
c["tranche"] = np.select(
    [c.SCORE == 0, (c.SCORE > 0) & (c.SCORE < s), c.SCORE >= s],
    ["Score = 0", "0 < Score < Seuil", "Score >= Seuil"], default="Non score")

print(f"\nNouveaux detenteurs entre {MIL_AVANT} et {MIL_APRES} : {len(nouveaux):,}")
print(f"Liste campagne : {len(c):,} | dans la liste modele (>= seuil) : "
      f"{(c.tranche=='Score >= Seuil').sum():,} ({(c.tranche=='Score >= Seuil').mean():.1%})")
if "CLIC" in c.columns:
    print(f"Cliqueurs : {int(c.CLIC.sum()):,} | cliqueurs devenus detenteurs : "
          f"{int(((c.CLIC == 1) & (c.souscripteur == 1)).sum()):,}")

print("\n--- 7.1 Conversion par tranche (B73:C75) ---")
t = (c.groupby("tranche").agg(cibles=("ID", "size"), souscripteurs=("souscripteur", "sum"))
     .reindex(["Score = 0", "0 < Score < Seuil", "Score >= Seuil", "Non score"])
     .fillna(0).astype(int))
t["conversion"] = t.souscripteurs / t.cibles
t.loc["Total"] = [t.cibles.sum(), t.souscripteurs.sum(), t.souscripteurs.sum() / t.cibles.sum()]
print(t.to_string(formatters={"conversion": "{:.2%}".format}))

sel = t.loc["Score >= Seuil", "cibles"] / t.loc["Total", "cibles"]
cap = t.loc["Score >= Seuil", "souscripteurs"] / max(t.loc["Total", "souscripteurs"], 1)
print(f"\n--- 7.2 ---\nTaux de selection : {sel:.1%} | Taux de capture : {cap:.1%} | "
      f"Lift : {cap/sel if sel else float('nan'):.2f} | "
      f"N souscripteurs : {int(t.loc['Total','souscripteurs']):,}")


In [ ]:
# --- 7.3 Courbe de gains par decile (population campagne scoree) ---
cs = c[c.SCORE.notna()].copy()
cs["decile"] = pd.qcut(cs.SCORE.rank(method="first", ascending=False), 10, labels=range(1, 11))

g = cs.groupby("decile", observed=True).agg(cibles=("ID", "size"),
                                            souscripteurs=("souscripteur", "sum"))
g["conversion"] = g.souscripteurs / g.cibles
print(g.to_string(formatters={"conversion": "{:.2%}".format}))
print(f"\n--- A coller en B92:C101 ---\n"
      f"{g[['cibles','souscripteurs']].to_string(index=False, header=False)}")


## Section 8 - Profiling


In [ ]:
base = bases[CUR]
if FILTRE_PARC:
    base = base.query(FILTRE_PARC)

cibles = base[base.ID.isin(scores[CUR].loc[scores[CUR].SCORE >= seuils[CUR], "ID"])]
detent = base[base.ID.isin(detenteurs[CUR])]

prof = pd.DataFrame({
    "Clients a cibler": [cibles[v].mean() for v in COLS_PROFIL.values()],
    "Vrais detenteurs": [detent[v].mean() for v in COLS_PROFIL.values()],
    "Parc de comptes":  [base[v].mean()   for v in COLS_PROFIL.values()],
}, index=list(COLS_PROFIL))
prof["Ecart cibles - parc"] = prof["Clients a cibler"] - prof["Parc de comptes"]

print(f"Effectifs - cibles : {len(cibles):,} | detenteurs : {len(detent):,} | parc : {len(base):,}")
print(prof.round(4).to_string())


## Backtest sans campagne

Acquisitions depuis un millesime = detenteurs du dernier millesime absents de celui-ci.


In [ ]:
dernier = list(MILLESIMES)[-1]
res = []
for m in list(MILLESIMES)[:-1]:
    acq = pd.DataFrame({"ID": list(detenteurs[dernier] - detenteurs[m])})
    a = acq.merge(scores[m][["ID", "SCORE"]], on="ID", how="left")
    s = seuils[m]
    res.append({"Millesime": m, "Acquisitions": len(a),
                "Score >= seuil": int((a.SCORE >= s).sum()),
                "Score < seuil":  int(((a.SCORE < s) & a.SCORE.notna()).sum()),
                "Absents des scores": int(a.SCORE.isna().sum())})
bt = pd.DataFrame(res).set_index("Millesime")
for k in ["Score >= seuil", "Score < seuil", "Absents des scores"]:
    bt[k + " %"] = bt[k] / bt["Acquisitions"]
print(f"Detenteurs de reference : {dernier}\n")
print(bt.to_string(formatters={k: "{:.1%}".format for k in bt.columns if k.endswith("%")}))
